In [2]:
import os
import sys
import random
from pathlib import Path

# Налаштування до імпорту TensorFlow
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

try:
    import tensorflow as tf
    from tensorflow import keras
except ModuleNotFoundError as exc:
    if exc.name != "tensorflow":
        raise

    raise SystemExit(
        "TensorFlow не встановлено в поточному середовищі.\n"
        f"Python: {sys.executable}\n"
        f"Версія: {sys.version.split()[0]}\n\n"
        "Команда встановлення в терміналі:\n"
        f'"{sys.executable}" -m pip install tensorflow\n\n'
        "Якщо для цієї версії Python немає сумісного пакета, "
        "використайте середовище Python 3.12.\n"
        "Після встановлення виберіть це середовище у VS Code "
        "і перезапустіть ядро Jupyter."
    ) from exc

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation, PillowWriter
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)


# ============================================================
# Параметри практичної роботи — варіант 7
# ============================================================

JOURNAL_NUMBER = 7

N_SAMPLES = 1000 * JOURNAL_NUMBER
N_CLASSES = JOURNAL_NUMBER + 2

N_FEATURES = 20
N_INFORMATIVE = 10
N_REDUNDANT = 5
N_REPEATED = 0
N_CLUSTERS_PER_CLASS = 2

SEED = 42

# Навчальна / валідаційна / тестова вибірки: 70 / 15 / 15 %
TEST_SIZE = 0.15
VAL_SIZE = 0.15

EPOCHS = 40
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# Працює як у .py-файлі, так і в Jupyter Notebook
BASE_DIR = (
    Path(__file__).resolve().parent
    if "__file__" in globals()
    else Path.cwd()
)

RESULTS_DIR = BASE_DIR / "results"

CLASS_LABELS = list(range(N_CLASSES))
CLASS_NAMES = [f"Клас {i}" for i in CLASS_LABELS]


# ============================================================
# Допоміжні функції
# ============================================================

def set_seed(seed):
    """Фіксація генераторів випадкових чисел."""
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()


def is_interactive_backend():
    """Перевірка можливості показувати інтерактивні вікна."""
    backend = matplotlib.get_backend().lower()

    non_interactive = {
        "agg", "pdf", "ps", "svg", "pgf", "cairo", "template"
    }

    return (
        backend not in non_interactive
        and "inline" not in backend
    )


# ============================================================
# 1. Генерація даних
# ============================================================

def generate_data():
    if N_CLASSES * N_CLUSTERS_PER_CLASS > 2 ** N_INFORMATIVE:
        raise ValueError(
            "Параметри не відповідають обмеженню: "
            "n_classes * n_clusters_per_class <= 2**n_informative."
        )

    if N_INFORMATIVE + N_REDUNDANT + N_REPEATED > N_FEATURES:
        raise ValueError(
            "Сума інформативних, надлишкових і повторюваних "
            "ознак перевищує загальну кількість ознак."
        )

    X, y = make_classification(
        n_samples=N_SAMPLES,
        n_features=N_FEATURES,
        n_informative=N_INFORMATIVE,
        n_redundant=N_REDUNDANT,
        n_repeated=N_REPEATED,
        n_classes=N_CLASSES,
        n_clusters_per_class=N_CLUSTERS_PER_CLASS,
        random_state=SEED,
    )

    if not np.isfinite(X).all():
        raise ValueError("Згенеровані ознаки містять NaN або нескінченність.")

    if len(np.unique(y)) != N_CLASSES:
        raise ValueError("Кількість згенерованих класів не відповідає умові.")

    print("\n1. ГЕНЕРАЦІЯ ДАНИХ")
    print(f"Варіант: {JOURNAL_NUMBER}")
    print(f"Кількість зразків: {N_SAMPLES}")
    print(f"Кількість класів: {N_CLASSES}")
    print(f"Кількість ознак: {N_FEATURES}")
    print(f"Розмір X: {X.shape}")
    print(f"Розмір y: {y.shape}")

    classes, counts = np.unique(y, return_counts=True)

    print("\nКількість зразків кожного класу:")
    for class_id, count in zip(classes, counts):
        print(f"Клас {class_id}: {count}")

    return X, y


def plot_class_distribution(y):
    classes, counts = np.unique(y, return_counts=True)

    fig, ax = plt.subplots(figsize=(9, 5))

    bars = ax.bar(
        classes,
        counts,
        color="#4C72B0",
        edgecolor="black",
    )

    ax.bar_label(bars, padding=3)
    ax.set_xticks(CLASS_LABELS)
    ax.set_title(f"Розподіл класів: {len(y)} зразків")
    ax.set_xlabel("Клас")
    ax.set_ylabel("Кількість зразків")
    ax.set_ylim(0, counts.max() * 1.15)
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(
        RESULTS_DIR / "class_distribution.png",
        dpi=150,
    )

    return fig


# ============================================================
# 2. Поділ і масштабування даних
# ============================================================

def prepare_data(X, y):
    X_temp, X_test, y_temp, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=SEED,
    )

    # Валідаційна вибірка — 15 % від початкового набору
    val_fraction = VAL_SIZE / (1.0 - TEST_SIZE)

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=val_fraction,
        stratify=y_temp,
        random_state=SEED,
    )

    scaler = StandardScaler()

    # Навчання масштабувальника лише на навчальній вибірці
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    y_train = y_train.astype(np.int32)
    y_val = y_val.astype(np.int32)
    y_test = y_test.astype(np.int32)

    print("\n2. ПІДГОТОВКА ДАНИХ")

    datasets = [
        ("Навчальна", X_train, y_train),
        ("Валідаційна", X_val, y_val),
        ("Тестова", X_test, y_test),
    ]

    for name, features, labels in datasets:
        if not np.isfinite(features).all():
            raise ValueError(f"{name} вибірка містить NaN або нескінченність.")

        counts = np.bincount(labels, minlength=N_CLASSES)

        print(f"{name}: X={features.shape}, y={labels.shape}")
        print(f"Розподіл класів: {counts.tolist()}")

    return X_train, X_val, X_test, y_train, y_val, y_test


# ============================================================
# 3. Побудова нейронної мережі
# ============================================================

def build_model():
    model = keras.Sequential(
        [
            keras.layers.Input(shape=(N_FEATURES,)),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(32, activation="relu"),
            keras.layers.Dense(N_CLASSES, activation="softmax"),
        ],
        name="classification_variant_7",
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


# ============================================================
# 4. Анімація навчання
# ============================================================

class LiveTrainingPlot(keras.callbacks.Callback):
    """Оновлення графіків після кожної епохи та збереження GIF."""

    def __init__(self, epochs, show):
        super().__init__()

        self.epochs = epochs
        self.show = show

        self.history = {
            "accuracy": [],
            "val_accuracy": [],
            "loss": [],
            "val_loss": [],
        }

        if self.show:
            plt.ion()

        self.fig, (self.ax_acc, self.ax_loss) = plt.subplots(
            1, 2, figsize=(12, 5)
        )

        self.line_acc, = self.ax_acc.plot(
            [], [], "o-", markersize=3, label="Навчальна"
        )
        self.line_val_acc, = self.ax_acc.plot(
            [], [], "s-", markersize=3, label="Валідаційна"
        )
        self.line_loss, = self.ax_loss.plot(
            [], [], "o-", markersize=3, label="Навчальна"
        )
        self.line_val_loss, = self.ax_loss.plot(
            [], [], "s-", markersize=3, label="Валідаційна"
        )

        self.ax_acc.set_title("Accuracy")
        self.ax_acc.set_ylabel("Accuracy")
        self.ax_acc.set_ylim(0, 1)

        self.ax_loss.set_title("Loss")
        self.ax_loss.set_ylabel("Cross-entropy")

        for ax in (self.ax_acc, self.ax_loss):
            ax.set_xlabel("Епоха")
            ax.set_xlim(0.5, epochs + 0.5)
            ax.grid(alpha=0.3)
            ax.legend()

        self.fig.suptitle("Навчання нейронної мережі")
        self.fig.tight_layout(rect=(0, 0, 1, 0.90))

        if self.show:
            plt.show(block=False)
            plt.pause(0.01)

    def redraw(self, n):
        epochs = np.arange(1, n + 1)
        history = self.history

        self.line_acc.set_data(
            epochs, history["accuracy"][:n]
        )
        self.line_val_acc.set_data(
            epochs, history["val_accuracy"][:n]
        )
        self.line_loss.set_data(
            epochs, history["loss"][:n]
        )
        self.line_val_loss.set_data(
            epochs, history["val_loss"][:n]
        )

        losses = history["loss"][:n] + history["val_loss"][:n]
        self.ax_loss.set_ylim(0, max(0.1, max(losses) * 1.1))

        self.fig.suptitle(
            f"Епоха {n}/{self.epochs}\n"
            f"accuracy={history['accuracy'][n - 1]:.4f}; "
            f"val_accuracy={history['val_accuracy'][n - 1]:.4f}; "
            f"loss={history['loss'][n - 1]:.4f}; "
            f"val_loss={history['val_loss'][n - 1]:.4f}"
        )

        return (
            self.line_acc,
            self.line_val_acc,
            self.line_loss,
            self.line_val_loss,
        )

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        for key in self.history:
            if key not in logs:
                raise RuntimeError(
                    f"Метрика '{key}' відсутня в результатах епохи."
                )
            self.history[key].append(float(logs[key]))

        self.redraw(epoch + 1)

        if self.show and plt.fignum_exists(self.fig.number):
            self.fig.canvas.draw_idle()
            plt.pause(0.01)

    def save_gif(self, path, fps=6):
        frames_count = len(self.history["loss"])

        if frames_count == 0:
            raise RuntimeError("Немає результатів навчання для анімації.")

        animation = FuncAnimation(
            self.fig,
            lambda frame: self.redraw(frame + 1),
            frames=frames_count,
            interval=1000 / fps,
            blit=False,
            repeat=False,
        )

        animation.save(
            str(path),
            writer=PillowWriter(fps=fps),
            dpi=80,
        )

        self.redraw(frames_count)
        print(f"Анімацію збережено: {path}")


# ============================================================
# 5. Підсумкові графіки й оцінювання
# ============================================================

def plot_final_history(history):
    epochs = np.arange(1, len(history["loss"]) + 1)

    fig, (ax_acc, ax_loss) = plt.subplots(
        1, 2, figsize=(12, 5)
    )

    ax_acc.plot(
        epochs, history["accuracy"], label="Навчальна"
    )
    ax_acc.plot(
        epochs, history["val_accuracy"], label="Валідаційна"
    )
    ax_acc.set_title("Accuracy")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.set_ylim(0, 1)

    ax_loss.plot(
        epochs, history["loss"], label="Навчальна"
    )
    ax_loss.plot(
        epochs, history["val_loss"], label="Валідаційна"
    )
    ax_loss.set_title("Loss")
    ax_loss.set_ylabel("Cross-entropy")

    for ax in (ax_acc, ax_loss):
        ax.set_xlabel("Епоха")
        ax.grid(alpha=0.3)
        ax.legend()

    fig.tight_layout()
    fig.savefig(
        RESULTS_DIR / "training_history.png",
        dpi=150,
    )

    return fig


def plot_confusion_matrix(cm, test_accuracy):
    fig, ax = plt.subplots(figsize=(10, 8))

    image = ax.imshow(cm, cmap="Blues")
    fig.colorbar(image, ax=ax)

    ax.set_xticks(CLASS_LABELS)
    ax.set_yticks(CLASS_LABELS)
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)

    ax.set_xlabel("Прогнозований клас")
    ax.set_ylabel("Істинний клас")
    ax.set_title(
        f"Матриця плутанини: accuracy = {test_accuracy:.4f}"
    )

    threshold = cm.max() / 2

    for row in range(N_CLASSES):
        for column in range(N_CLASSES):
            ax.text(
                column,
                row,
                str(cm[row, column]),
                ha="center",
                va="center",
                color=(
                    "white"
                    if cm[row, column] > threshold
                    else "black"
                ),
            )

    fig.tight_layout()
    fig.savefig(
        RESULTS_DIR / "confusion_matrix.png",
        dpi=150,
    )

    return fig


# ============================================================
# Виконання практичної роботи
# ============================================================

def main():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    set_seed(SEED)

    show = is_interactive_backend()

    print(f"Python: {sys.version.split()[0]}")
    print(f"TensorFlow: {tf.__version__}")
    print(f"Matplotlib backend: {matplotlib.get_backend()}")

    # 1. Генерація даних
    X, y = generate_data()
    plot_class_distribution(y)

    if show:
        plt.show(block=False)
        plt.pause(0.01)

    # 2. Підготовка
    (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
    ) = prepare_data(X, y)

    # 3. Модель
    print("\n3. НЕЙРОННА МЕРЕЖА")
    model = build_model()
    model.summary()

    # 4. Навчання
    print("\n4. НАВЧАННЯ")
    live_plot = LiveTrainingPlot(EPOCHS, show)

    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=[live_plot],
        verbose=2,
    )

    live_plot.save_gif(
        RESULTS_DIR / "training_animation.gif"
    )

    plot_final_history(live_plot.history)

    # 5. Оцінювання
    print("\n5. ОЦІНЮВАННЯ НА ТЕСТОВІЙ ВИБІРЦІ")

    probabilities = model.predict(X_test, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    test_accuracy = accuracy_score(y_test, y_pred)

    report = classification_report(
        y_test,
        y_pred,
        labels=CLASS_LABELS,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=CLASS_LABELS,
    )

    print(f"\nTest accuracy: {test_accuracy:.4f}")
    print("\nЗвіт класифікації:")
    print(report)
    print("Матриця плутанини:")
    print(cm)

    plot_confusion_matrix(cm, test_accuracy)

    report_path = RESULTS_DIR / "classification_report.txt"

    with report_path.open("w", encoding="utf-8") as file:
        file.write(f"Практична 8. Варіант {JOURNAL_NUMBER}\n")
        file.write(f"Кількість зразків: {N_SAMPLES}\n")
        file.write(f"Кількість класів: {N_CLASSES}\n")
        file.write(
            f"Навчальна вибірка: {len(y_train)}\n"
            f"Валідаційна вибірка: {len(y_val)}\n"
            f"Тестова вибірка: {len(y_test)}\n"
        )
        file.write(f"\nTest accuracy: {test_accuracy:.4f}\n\n")
        file.write(report)
        file.write("\nМатриця плутанини:\n")
        file.write(np.array2string(cm))
        file.write("\n")

    print(f"\nРезультати збережено: {RESULTS_DIR}")

    plt.ioff()

    if show:
        plt.show(block=True)
    elif "inline" in matplotlib.get_backend().lower():
        # Показ статичних графіків у Jupyter
        plt.show()
        plt.close("all")
    else:
        plt.close("all")


if __name__ == "__main__":
    main()

SystemExit: TensorFlow не встановлено в поточному середовищі.
Python: c:\Users\Вейдер\AppData\Local\Python\pythoncore-3.14-64\python.exe
Версія: 3.14.3

Команда встановлення в терміналі:
"c:\Users\Вейдер\AppData\Local\Python\pythoncore-3.14-64\python.exe" -m pip install tensorflow

Якщо для цієї версії Python немає сумісного пакета, використайте середовище Python 3.12.
Після встановлення виберіть це середовище у VS Code і перезапустіть ядро Jupyter.

C:\Users\Вейдер\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
